# 10年定着予測 - 特徴量エンジニアリング アブレーション（テキスト・部署時系列・ランク・粒度細分化）

**目的**: `15_enriched_feature_engineering`で、テキストTF-IDF・部署時系列・ランク特徴量・四半期粒度/加速度の
4ブロック（計約90列）をまとめて追加したところ、14_（既存315列、Public 0.563076）よりPublicスコアが
悪化した（0.583434）。特徴量重要度では部署時系列ブロックが最上位を維持していた一方、ランク特徴量は
ほぼ寄与していなかったため、**4ブロックを個別に切り分けて、どれが真に効いているかを特定する**。

## アブレーション設定（7パターン）

| 設定名 | 内容 |
|---|---|
| `baseline` | 既存315列のみ（14_相当の再現） |
| `A_only` | 既存315列 + テキストTF-IDF（45列） |
| `B_only` | 既存315列 + 部署時系列（4列） |
| `C_only` | 既存315列 + ランク特徴量（3列） |
| `D_only` | 既存315列 + 四半期/加速度（30列） |
| `B_plus_D` | 既存315列 + 部署時系列 + 四半期/加速度（重要度が高かった2ブロック） |
| `all_except_C` | 既存315列 + A + B + D（ランク特徴量だけ除外） |

特徴量エンジニアリングのコード自体は`15_`（部署Target Encodingのリークを修正済み）から完全に流用する。

## モデリング方針

`14_`の実験Aで確認した通り、`common/catboost/`ラッパーを使わない自前のCatBoost + Optuna
（単一時系列ホールドアウト検証）は、AutoGluonのCatBoostとPublicスコアで実質同一の性能に達することが分かっている。
本ノートブックもAutoGluonは使わず、**同じ自前CatBoost + Optunaパイプラインを7パターン分ループで実行する**。

## 実行環境
Google Colab（GPU: T4）を想定。

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 26.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 30.1 MB/s eta 0:00:00


In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Aug  9 01:02:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "16_feature_ablation"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Output Directory: {OUTPUT_DIR}")

[2026-08-09 01:03:27] [INFO] === [16_feature_ablation] 実験開始 ===


INFO:16_feature_ablation:=== [16_feature_ablation] 実験開始 ===


[2026-08-09 01:03:28] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


INFO:16_feature_ablation:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-09 01:03:31] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:16_feature_ablation:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-09 01:03:31] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:16_feature_ablation:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-09 01:03:31] [INFO] 定着率: 0.5647


INFO:16_feature_ablation:定着率: 0.5647


[2026-08-09 01:03:31] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:16_feature_ablation:Train IDs: 2761, Test IDs: 2502


## 0.5 検証用ホールドアウトのID集合を先に確定する（重要）

**設計上の注意（本ノートブックで新たに対応した点）**: 部署IDのような小規模グループ（1グループ平均6名）に対する
Target Encodingを、Train全体（チューニング期間の社員を含む）でfitしてしまうと、チューニング期間の社員の特徴量に
「同じ部署の他のチューニング期間社員のラベル」が漏れ込む（KFoldのfold分けが日付とは無関係なため）。
検証の結果、この漏れは実際に発生しており、チューニング期間内で特徴量と正解ラベルの相関が
0.47近くまで不自然に高くなることを確認した（正しく学習期間のみでfitすると0.1程度が妥当）。

これを防ぐため、**入社日でソートした際の「学習期間（先頭80%）」のIDを先に確定し、
Target Encoding等リークしうる集計は必ずこの学習期間のIDのみでfitする**。

In [7]:
train_persona_sorted_for_split = train_persona.copy()
train_persona_sorted_for_split["入社日"] = pd.to_datetime(train_persona_sorted_for_split["入社日"])
train_persona_sorted_for_split = train_persona_sorted_for_split.sort_values("入社日")

_split_point = int(len(train_persona_sorted_for_split) * 0.8)
TRAIN_PERIOD_IDS = set(train_persona_sorted_for_split.iloc[:_split_point][ID_COL])
TUNING_PERIOD_IDS = set(train_persona_sorted_for_split.iloc[_split_point:][ID_COL])

logger.info(f"学習期間: {len(TRAIN_PERIOD_IDS)}件, 検証期間: {len(TUNING_PERIOD_IDS)}件")
logger.info("以降、Target Encoding等はTRAIN_PERIOD_IDSのみでfitする")

[2026-08-09 01:03:31] [INFO] 学習期間: 2208件, 検証期間: 553件


INFO:16_feature_ablation:学習期間: 2208件, 検証期間: 553件


[2026-08-09 01:03:31] [INFO] 以降、Target Encoding等はTRAIN_PERIOD_IDSのみでfitする


INFO:16_feature_ablation:以降、Target Encoding等はTRAIN_PERIOD_IDSのみでfitする


## 1. 基本特徴量（13_/14_と完全に同一）

これまでの実験で確立した最良の基本特徴量セットをそのまま流用する。

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_/13_/14_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了")

✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了


In [9]:
def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding

    [15_での修正] fit_ids（時系列splitの学習期間のID集合）に含まれる社員のみでマップを構築する。
    Train全体（検証期間の社員を含む）でfitすると、部署のような小規模グループ経由で
    検証期間の社員のラベルが互いに漏れ込むリークが発生するため（本ノートブックで検出・修正）。
    """
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    # 学習期間内のみでKFold-OOFを計算（学習に使う特徴量が自分のラベルを見ないようにする）
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    # 学習期間全体でのマップ（検証期間の社員・Testにはこのマップを直接適用。学習に使われないためOOF不要）
    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        "社員ID": train_persona["社員ID"].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        "社員ID": test_persona["社員ID"].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out, mapping_full, global_mean


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_/13_/14_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_/13_/14_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ 基本特徴量関数定義完了")

✅ 基本特徴量関数定義完了


In [10]:
logger.info("-" * 60)
logger.info("基本特徴量生成開始")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_dept_te, test_dept_te, dept_te_map, GLOBAL_MEAN = create_department_target_encoding(
    train_persona, test_persona, y_train, fit_ids=TRAIN_PERIOD_IDS, seed=SEED, n_splits=5, smoothing=10
)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("基本特徴量生成完了")

[2026-08-09 01:03:32] [INFO] ------------------------------------------------------------


INFO:16_feature_ablation:------------------------------------------------------------


[2026-08-09 01:03:32] [INFO] 基本特徴量生成開始


INFO:16_feature_ablation:基本特徴量生成開始


[2026-08-09 01:03:32] [INFO] ------------------------------------------------------------


INFO:16_feature_ablation:------------------------------------------------------------


[2026-08-09 01:10:48] [INFO] 基本特徴量生成完了


INFO:16_feature_ablation:基本特徴量生成完了


In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
text_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-09 01:10:49] [INFO] Persona単位の基本特徴量を生成中...


INFO:16_feature_ablation:Persona単位の基本特徴量を生成中...


[2026-08-09 01:10:49] [INFO] Persona単位の基本特徴量処理完了


INFO:16_feature_ablation:Persona単位の基本特徴量処理完了


## 2. 新規特徴量A: テキストのTF-IDF + SVD特徴量

入社時メモ・上司/同僚フィードバックの3つのテキストについて、文字n-gram（2〜4文字）のTF-IDFを計算し、
TruncatedSVDで次元圧縮する。ボキャブラリ（`TfidfVectorizer`）とSVDはTrainのみでfitし、Testはtransformのみ行う
（リークを避けるため）。`algorithm='arpack'`を指定し、数値的な警告を避ける。

In [12]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, seed=42):
    """文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=3)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

logger.info("テキストTF-IDF+SVD特徴量を生成中...")
text_feature_frames_train, text_feature_frames_test = [], []
for col in ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]:
    tr_tfidf, te_tfidf, explained_var = create_tfidf_svd_features(
        train_persona, test_persona, col, max_features=300, n_components=15, seed=SEED
    )
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    text_feature_frames_train.append(tr_tfidf)
    text_feature_frames_test.append(te_tfidf)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-09 01:10:49] [INFO] テキストTF-IDF+SVD特徴量を生成中...


INFO:16_feature_ablation:テキストTF-IDF+SVD特徴量を生成中...


[2026-08-09 01:10:51] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:16_feature_ablation:入社時メモ: SVD累積寄与率=0.760


[2026-08-09 01:10:55] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:16_feature_ablation:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-09 01:10:57] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.422


INFO:16_feature_ablation:同僚からのフィードバック: SVD累積寄与率=0.422


[2026-08-09 01:10:57] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:16_feature_ablation:テキストTF-IDF+SVD特徴量生成完了


## 3. 新規特徴量B: 部署の時系列変化特徴量

`create_department_target_encoding`で構築した部署別TEマップ（辞書）を再利用し、月次データから
「最終部署」「在籍した全部署の平均」を計算して同じマップでTarget Encodingする
（マップ自体は初期部署IDとTrainの目的変数のKFold+スムージングで作成済みなので、
別の部署カラムに適用してもリークしない）。

In [13]:
def create_department_timeseries_features(monthly_df, employee_ids, dept_te_map, global_mean):
    """最終部署TE・経験部署TE平均・初期→最終のTE変化・初期最終部署の同一フラグ"""
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        depts = emp_data["部署ID"].dropna().unique()
        initial_dept = emp_data["部署ID"].iloc[0] if len(emp_data) > 0 else None
        final_dept = emp_data["部署ID"].iloc[-1] if len(emp_data) > 0 else None

        initial_te = dept_te_map.get(initial_dept, global_mean)
        final_te = dept_te_map.get(final_dept, global_mean)
        avg_te = np.mean([dept_te_map.get(d, global_mean) for d in depts]) if len(depts) > 0 else global_mean

        features["最終部署_target_enc"] = final_te
        features["経験部署_target_enc_avg"] = avg_te
        features["部署te_diff_final_minus_initial"] = final_te - initial_te
        features["初期最終部署_同一フラグ"] = int(initial_dept == final_dept)
        features_list.append(features)
    return pd.DataFrame(features_list)

logger.info("部署の時系列変化特徴量を生成中...")
train_dept_ts = create_department_timeseries_features(train_monthly, train_ids, dept_te_map, GLOBAL_MEAN)
test_dept_ts = create_department_timeseries_features(test_monthly, test_ids, dept_te_map, GLOBAL_MEAN)
logger.info(f"Train: {train_dept_ts.shape}, Test: {test_dept_ts.shape}")
logger.info(f"初期最終部署_同一フラグ 分布: {train_dept_ts['初期最終部署_同一フラグ'].value_counts().to_dict()}")

[2026-08-09 01:10:57] [INFO] 部署の時系列変化特徴量を生成中...


INFO:16_feature_ablation:部署の時系列変化特徴量を生成中...


[2026-08-09 01:11:26] [INFO] Train: (2761, 5), Test: (2502, 5)


INFO:16_feature_ablation:Train: (2761, 5), Test: (2502, 5)


[2026-08-09 01:11:26] [INFO] 初期最終部署_同一フラグ 分布: {1: 2121, 0: 640}


INFO:16_feature_ablation:初期最終部署_同一フラグ 分布: {1: 2121, 0: 640}


## 4. 新規特徴量C: 相対化・ランク特徴量

既存の「グループ平均からの偏差」に加え、パーセンタイル順位を追加する。分布形状に頑健で、
Train/Testそれぞれの母集団内で計算する（Train統計量をTestに適用するのではなく、
各データセット自身の分布内での相対順位を使うため、Train/Testの給与水準シフトの影響を受けにくい）。

In [14]:
def create_rank_features(df, group_cols, value_cols):
    """グループ内パーセンタイル順位（そのデータセット自身の分布内で計算）"""
    out = pd.DataFrame(index=df.index)
    out[ID_COL] = df[ID_COL].values
    for group_col in group_cols:
        for value_col in value_cols:
            if value_col in df.columns and group_col in df.columns:
                out[f"{value_col}_rank_by_{group_col}"] = df.groupby(group_col)[value_col].rank(pct=True)
    return out

logger.info("相対化・ランク特徴量を生成中...")
# persona単体の初任給ランク（persona情報のみで完結）
rank_group_cols_persona = ["初期等級", "入社四半期", "初期職種"]
train_rank_persona = create_rank_features(train_persona, rank_group_cols_persona, ["初任給_円"])
test_rank_persona = create_rank_features(test_persona, rank_group_cols_persona, ["初任給_円"])

logger.info(f"Train persona rank: {train_rank_persona.shape}, Test persona rank: {test_rank_persona.shape}")

[2026-08-09 01:11:26] [INFO] 相対化・ランク特徴量を生成中...


INFO:16_feature_ablation:相対化・ランク特徴量を生成中...


[2026-08-09 01:11:26] [INFO] Train persona rank: (2761, 4), Test persona rank: (2502, 4)


INFO:16_feature_ablation:Train persona rank: (2761, 4), Test persona rank: (2502, 4)


## 5. 新規特徴量D: 月次データの粒度細分化（四半期）+ 加速度特徴量

0-23ヶ月を4つの半期（Q1:0-5, Q2:6-11, Q3:12-17, Q4:18-23）に分割し、各期間の平均を計算する。
さらに「変化の加速度」= (Q4-Q3の変化幅) - (Q2-Q1の変化幅)を計算し、後半での変化の加速/減速を捉える。

In [15]:
def create_quarterly_features(monthly_df, employee_ids, metrics):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

QUARTERLY_METRICS = ["残業時間", "欠勤日数", "月例給与_円", "360度評価_親和度", "有給取得日数", "研修時間"]

logger.info("月次データの四半期粒度・加速度特徴量を生成中...")
train_quarterly = create_quarterly_features(train_monthly, train_ids, QUARTERLY_METRICS)
test_quarterly = create_quarterly_features(test_monthly, test_ids, QUARTERLY_METRICS)
logger.info(f"Train: {train_quarterly.shape}, Test: {test_quarterly.shape}")

[2026-08-09 01:11:26] [INFO] 月次データの四半期粒度・加速度特徴量を生成中...


INFO:16_feature_ablation:月次データの四半期粒度・加速度特徴量を生成中...


[2026-08-09 01:12:48] [INFO] Train: (2761, 31), Test: (2502, 31)


INFO:16_feature_ablation:Train: (2761, 31), Test: (2502, 31)


## 6. 全特徴量の統合（基本特徴量 + 新規4ブロック）

In [16]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")
train_features = train_features.merge(train_advanced_stats, on=ID_COL, how="left")
train_features = train_features.merge(train_cluster, on=ID_COL, how="left")
train_features = train_features.merge(train_dept_te, on=ID_COL, how="left")
train_features = train_features.merge(train_eda_feats, on=ID_COL, how="left")
train_features = train_features.merge(train_mgr, on=ID_COL, how="left")
# --- 新規4ブロック ---
for tfidf_df in text_feature_frames_train:
    train_features = train_features.merge(tfidf_df, on=ID_COL, how="left")
train_features = train_features.merge(train_dept_ts, on=ID_COL, how="left")
train_features = train_features.merge(train_rank_persona, on=ID_COL, how="left")
train_features = train_features.merge(train_quarterly, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")
test_features = test_features.merge(test_advanced_stats, on=ID_COL, how="left")
test_features = test_features.merge(test_cluster, on=ID_COL, how="left")
test_features = test_features.merge(test_dept_te, on=ID_COL, how="left")
test_features = test_features.merge(test_eda_feats, on=ID_COL, how="left")
test_features = test_features.merge(test_mgr, on=ID_COL, how="left")
for tfidf_df in text_feature_frames_test:
    test_features = test_features.merge(tfidf_df, on=ID_COL, how="left")
test_features = test_features.merge(test_dept_ts, on=ID_COL, how="left")
test_features = test_features.merge(test_rank_persona, on=ID_COL, how="left")
test_features = test_features.merge(test_quarterly, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-09 01:12:49] [INFO] ------------------------------------------------------------


INFO:16_feature_ablation:------------------------------------------------------------


[2026-08-09 01:12:49] [INFO] 特徴量の統合


INFO:16_feature_ablation:特徴量の統合


[2026-08-09 01:12:49] [INFO] ------------------------------------------------------------


INFO:16_feature_ablation:------------------------------------------------------------


[2026-08-09 01:12:49] [INFO] Train: (2761, 397), Test: (2502, 397)


INFO:16_feature_ablation:Train: (2761, 397), Test: (2502, 397)


In [17]:
logger.info("職種別の乖離特徴量を生成中...")
# [15_での修正] グループ平均は学習期間(TRAIN_PERIOD_IDS)のみで計算する（職種・入社区分・等級は
# グループサイズが大きく漏れの影響は軽微だが、部署の教訓を踏まえ一貫性のため統一する）
_train_period_features = train_features[train_features[ID_COL].isin(TRAIN_PERIOD_IDS)]

job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}

for df in [train_features, test_features]:
    for m in job_dev_metrics:
        job_mean_series = df["初期職種"].map(job_means[m])
        df[f"{m}_job_deviation"] = df[m] - job_mean_series
    df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
    df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)

logger.info("給与の相対化特徴量を生成中...")
grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

for df in [train_features, test_features]:
    df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
    df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
    df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

logger.info(f"派生特徴量生成後 Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-09 01:12:49] [INFO] 職種別の乖離特徴量を生成中...


INFO:16_feature_ablation:職種別の乖離特徴量を生成中...


[2026-08-09 01:12:49] [INFO] 給与の相対化特徴量を生成中...


INFO:16_feature_ablation:給与の相対化特徴量を生成中...


[2026-08-09 01:12:49] [INFO] 派生特徴量生成後 Train: (2761, 405), Test: (2502, 405)


INFO:16_feature_ablation:派生特徴量生成後 Train: (2761, 405), Test: (2502, 405)


In [18]:
drop_cols = [
    "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
    "初期部署ID", "初期等級", "最終学歴", "前職職種",
]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

train_features = train_features.set_index(ID_COL)
test_features = test_features.set_index(ID_COL)

logger.info(f"最終特徴量数: {train_features.shape[1]} (Train: {train_features.shape}, Test: {test_features.shape})")

[2026-08-09 01:12:49] [INFO] 最終特徴量数: 397 (Train: (2761, 397), Test: (2502, 397))


INFO:16_feature_ablation:最終特徴量数: 397 (Train: (2761, 397), Test: (2502, 397))


## 7. 時系列ホールドアウトの作成（13_/14_と同一split）

In [19]:
target_series = train_persona.set_index(ID_COL)[TARGET_COL]

train_features_sorted = train_features.sort_values("入社日")
y_sorted = target_series.loc[train_features_sorted.index]

split_point = int(len(train_features_sorted) * 0.8)
ag_train_data = train_features_sorted.iloc[:split_point].copy()
ag_tuning_data = train_features_sorted.iloc[split_point:].copy()
ag_train_data[TARGET_COL] = y_sorted.iloc[:split_point].values
ag_tuning_data[TARGET_COL] = y_sorted.iloc[split_point:].values

logger.info(f"train_data: {ag_train_data.shape} (入社日 {ag_train_data['入社日'].min().date()} 〜 {ag_train_data['入社日'].max().date()})")
logger.info(f"tuning_data: {ag_tuning_data.shape} (入社日 {ag_tuning_data['入社日'].min().date()} 〜 {ag_tuning_data['入社日'].max().date()})")

[2026-08-09 01:12:50] [INFO] train_data: (2208, 398) (入社日 2011-04-01 〜 2013-04-01)


INFO:16_feature_ablation:train_data: (2208, 398) (入社日 2011-04-01 〜 2013-04-01)


[2026-08-09 01:12:50] [INFO] tuning_data: (553, 398) (入社日 2013-04-01 〜 2014-03-01)


INFO:16_feature_ablation:tuning_data: (553, 398) (入社日 2013-04-01 〜 2014-03-01)


## 8. アブレーション設計

15_で4ブロックをまとめて追加した結果、14_（既存315列）よりPublicスコアが悪化した（0.563076→0.583434）。
特徴量重要度では部署時系列ブロック（B）が最上位を維持していた一方、ランク特徴量（C）はほぼ寄与していなかった。
**どのブロックが真に効いているのかを切り分けるため**、以下7パターンで同一のCatBoost + Optunaパイプライン
（14_実験Aと同一、単一時系列ホールドアウト検証）を実行し、検証Log Lossを比較する。

| 設定名 | 内容 |
|---|---|
| `baseline` | 既存315列のみ（14_相当の再現） |
| `A_only` | 既存315列 + テキストTF-IDF（45列） |
| `B_only` | 既存315列 + 部署時系列（4列） |
| `C_only` | 既存315列 + ランク特徴量（3列） |
| `D_only` | 既存315列 + 四半期/加速度（30列） |
| `B_plus_D` | 既存315列 + 部署時系列 + 四半期/加速度（重要度が高かった2ブロック） |
| `all_except_C` | 既存315列 + A + B + D（ランク特徴量だけ除外） |

各設定について、Optuna探索（試行数は7パターン合計の実行時間を考慮し25回）→ベストパラメータで深い再学習
→検証Log Loss算出→Test予測・提出ファイル保存、まで自動で行う。

In [20]:
TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

FEATURE_GROUPS = {
    "A_tfidf": [f"{col}_tfidf_svd_{i}" for col in TEXT_COLS for i in range(15)],
    "B_dept_ts": ["最終部署_target_enc", "経験部署_target_enc_avg", "部署te_diff_final_minus_initial", "初期最終部署_同一フラグ"],
    "C_rank": ["初任給_円_rank_by_初期等級", "初任給_円_rank_by_入社四半期", "初任給_円_rank_by_初期職種"],
    "D_quarterly": [f"{m}_{q}_mean" for m in QUARTERLY_METRICS for q in ["q1", "q2", "q3", "q4"]] + [f"{m}_acceleration" for m in QUARTERLY_METRICS],
}

_all_new_cols = set().union(*FEATURE_GROUPS.values())
BASE_FEATURE_COLS = [c for c in train_features.columns if c != "入社日" and c not in _all_new_cols]

logger.info(f"BASE_FEATURE_COLS: {len(BASE_FEATURE_COLS)}列")
for gname, cols in FEATURE_GROUPS.items():
    missing = [c for c in cols if c not in train_features.columns]
    logger.info(f"{gname}: {len(cols)}列 (欠落: {len(missing)})")

ABLATION_CONFIGS = [
    ("baseline", []),
    ("A_only", ["A_tfidf"]),
    ("B_only", ["B_dept_ts"]),
    ("C_only", ["C_rank"]),
    ("D_only", ["D_quarterly"]),
    ("B_plus_D", ["B_dept_ts", "D_quarterly"]),
    ("all_except_C", ["A_tfidf", "B_dept_ts", "D_quarterly"]),
]

[2026-08-09 01:12:50] [INFO] BASE_FEATURE_COLS: 314列


INFO:16_feature_ablation:BASE_FEATURE_COLS: 314列


[2026-08-09 01:12:50] [INFO] A_tfidf: 45列 (欠落: 0)


INFO:16_feature_ablation:A_tfidf: 45列 (欠落: 0)


[2026-08-09 01:12:50] [INFO] B_dept_ts: 4列 (欠落: 0)


INFO:16_feature_ablation:B_dept_ts: 4列 (欠落: 0)


[2026-08-09 01:12:50] [INFO] C_rank: 3列 (欠落: 0)


INFO:16_feature_ablation:C_rank: 3列 (欠落: 0)


[2026-08-09 01:12:50] [INFO] D_quarterly: 30列 (欠落: 0)


INFO:16_feature_ablation:D_quarterly: 30列 (欠落: 0)


## 9. アブレーション実行

各設定で「Optuna探索→深い再学習→検証→Test予測→保存」を行う関数を定義し、7パターンをループで実行する。

In [21]:
def run_ablation_config(config_name, extra_groups, n_trials=25):
    feature_cols = BASE_FEATURE_COLS + [c for g in extra_groups for c in FEATURE_GROUPS[g]]
    cat_cols = [c for c in feature_cols if train_features[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000,
            "random_seed": SEED,
            "verbose": False,
            "cat_features": cat_cols,
            "early_stopping_rounds": 50,
            "task_type": "GPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        preds = model.predict_proba(X_va)[:, 1]
        return log_loss(y_va, preds)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=cat_cols, early_stopping_rounds=100, task_type="GPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)

    val_preds = final_model.predict_proba(X_va)[:, 1]
    val_score = log_loss(y_va, val_preds)

    X_test = test_features[feature_cols].fillna(-999)
    test_preds = final_model.predict_proba(X_test)[:, 1]
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_name}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    logger.info(f"[{config_name}] n_features={len(feature_cols)}, val_score={val_score:.6f}, best_params={best_params}")
    return {
        "config": config_name,
        "n_features": len(feature_cols),
        "val_score": val_score,
        "submission_path": str(sub_path),
    }

logger.info("=" * 60)
logger.info("アブレーション実行開始")
logger.info("=" * 60)

ablation_results = []
for config_name, extra_groups in ABLATION_CONFIGS:
    logger.info(f"--- 設定: {config_name} (追加ブロック: {extra_groups}) ---")
    result = run_ablation_config(config_name, extra_groups, n_trials=25)
    ablation_results.append(result)

ablation_df = pd.DataFrame(ablation_results).sort_values("val_score").reset_index(drop=True)

[2026-08-09 01:12:50] [INFO] ============================================================


INFO:16_feature_ablation:============================================================


[2026-08-09 01:12:50] [INFO] アブレーション実行開始


INFO:16_feature_ablation:アブレーション実行開始


[2026-08-09 01:12:50] [INFO] ============================================================


...

INFO:16_feature_ablation:[B_plus_D] n_features=348, val_score=0.574837, best_params={'depth': 8, 'learning_rate': 0.03673757945485905, 'l2_leaf_reg': 0.059018380990511494, 'border_count': 208, 'bagging_temperature': 0.6898904233496919, 'random_strength': 9.870356316661441}


[2026-08-09 02:44:13] [INFO] --- 設定: all_except_C (追加ブロック: ['A_tfidf', 'B_dept_ts', 'D_quarterly']) ---


INFO:16_feature_ablation:--- 設定: all_except_C (追加ブロック: ['A_tfidf', 'B_dept_ts', 'D_quarterly']) ---


[2026-08-09 02:52:50] [INFO] [all_except_C] n_features=393, val_score=0.562427, best_params={'depth': 8, 'learning_rate': 0.06161626405379123, 'l2_leaf_reg': 9.685201972048759, 'border_count': 208, 'bagging_temperature': 0.7533273171325099, 'random_strength': 9.896832814309462}


INFO:16_feature_ablation:[all_except_C] n_features=393, val_score=0.562427, best_params={'depth': 8, 'learning_rate': 0.06161626405379123, 'l2_leaf_reg': 9.685201972048759, 'border_count': 208, 'bagging_temperature': 0.7533273171325099, 'random_strength': 9.896832814309462}


## 10. 結果の比較

各設定の検証Log Lossを14_（既存315列、0.552597）・15_（全4ブロック、0.570387）と比較する。

In [22]:
logger.info("=" * 60)
logger.info("アブレーション結果サマリ（検証Log Loss、小さいほど良い）")
logger.info("=" * 60)
logger.info("\n" + ablation_df.to_string())

print("\n■ アブレーション結果（検証Log Loss順）:")
print(ablation_df.to_string(index=False))
print(f"\n■ (参考) 14_ baseline相当: 0.552597")
print(f"■ (参考) 15_ 全4ブロック: 0.570387")

best_config = ablation_df.iloc[0]
print(f"\n■ 最良設定: {best_config['config']} (val_score={best_config['val_score']:.6f})")
print(f"■ 提出ファイル: {best_config['submission_path']}")

ablation_df

[2026-08-09 02:52:51] [INFO] ============================================================


INFO:16_feature_ablation:============================================================


[2026-08-09 02:52:51] [INFO] アブレーション結果サマリ（検証Log Loss、小さいほど良い）


INFO:16_feature_ablation:アブレーション結果サマリ（検証Log Loss、小さいほど良い）


[2026-08-09 02:52:51] [INFO] ============================================================


INFO:16_feature_ablation:============================================================


[2026-08-09 02:52:51] [INFO] 
         config  n_features  val_score                                                                                        submission_path
0        A_only         359   0.529646        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_A_only.csv
1      baseline         314   0.547645      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_baseline.csv
2        C_only         317   0.547661        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_C_only.csv
3        D_only         344   0.548956        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_D_only.csv
4  all_except_C         393   0.562427  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_all_except_C.csv
5        B_only         318   0.571729        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feat

INFO:16_feature_ablation:
         config  n_features  val_score                                                                                        submission_path
0        A_only         359   0.529646        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_A_only.csv
1      baseline         314   0.547645      /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_baseline.csv
2        C_only         317   0.547661        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_C_only.csv
3        D_only         344   0.548956        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_D_only.csv
4  all_except_C         393   0.562427  /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_all_except_C.csv
5        B_only         318   0.571729        /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_


■ アブレーション結果（検証Log Loss順）:
      config  n_features  val_score                                                                                       submission_path
      A_only         359   0.529646       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_A_only.csv
    baseline         314   0.547645     /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_baseline.csv
      C_only         317   0.547661       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_C_only.csv
      D_only         344   0.548956       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_D_only.csv
all_except_C         393   0.562427 /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_all_except_C.csv
      B_only         318   0.571729       /content/drive/MyDrive/jaggle_2026/data/output/20260809/20260809_16_feature_ablation_B_only.csv
    B_p

,config,n_features,val_score,submission_path
0,A_only,359,0.529646,/content/drive/MyDrive/jaggle_2026/data/output...
1,baseline,314,0.547645,/content/drive/MyDrive/jaggle_2026/data/output...
2,C_only,317,0.547661,/content/drive/MyDrive/jaggle_2026/data/output...
3,D_only,344,0.548956,/content/drive/MyDrive/jaggle_2026/data/output...
4,all_except_C,393,0.562427,/content/drive/MyDrive/jaggle_2026/data/output...
5,B_only,318,0.571729,/content/drive/MyDrive/jaggle_2026/data/output...
6,B_plus_D,348,0.574837,/content/drive/MyDrive/jaggle_2026/data/output...


## 11. まとめ・次のアクション

このアブレーションにより、以下を切り分けられる。

1. **各ブロック単体（A_only, B_only, C_only, D_only）が`baseline`より改善するか**
   → 改善するブロックだけが「真に効く」候補。
2. **`B_plus_D`（重要度が高かった2ブロック）が`baseline`を上回るか**
   → 上回れば、部署時系列と四半期/加速度の組み合わせが有望。
3. **`all_except_C`が15_（全4ブロック）より改善するか**
   → ランク特徴量を除くだけで15_の悪化が解消されるなら、Cが主な悪化要因だったと言える。

検証Log Lossが最も良かった設定の提出ファイルをKaggleに提出し、Publicスコアで最終確認する。
結果が出たら`submit_result_report.md`に追記し、今後はこのアブレーションで判明した「効くブロックのみ」を
標準の特徴量セットとして採用する。